# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL for the FAIR² dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
print(f"Dataset ID: {dataset.metadata.id}")
print(f"Title: {getattr(dataset.metadata, 'name', '')}")
print(f"Description: {getattr(dataset.metadata, 'description', '')}")
print(f"Published: {getattr(dataset.metadata, 'datePublished', '')}")
print(f"License: {getattr(dataset.metadata, 'license', '')}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s. This is helpful for referencing specific parts of the dataset in subsequent steps.

In [ ]:
# List all available record sets by their @id and name (if available)
print("Record Sets in the Dataset:")
for record_set in dataset.metadata.recordSet:
    print(f"@id: {record_set['@id']}, name: {record_set.get('name', '')}")

# Explore fields in each record set
for record_set in dataset.metadata.recordSet:
    print(f"\nFields for RecordSet {record_set['@id']}:")
    if 'field' in record_set:
        for field in record_set['field']:
            print(f"  Field @id: {field['@id']}, name: {field.get('name', '')}, dataType: {field.get('dataType', '')}")

## 3. Data Extraction
Load data from the main record set (table) into a DataFrame for analysis. Entities are referenced by their `@id` as per the Croissant schema.

In [ ]:
# List of RecordSet @ids
record_sets_ids = [rs['@id'] for rs in dataset.metadata.recordSet]
print(f"RecordSet IDs: {record_sets_ids}")

dataframes = {}
# Load data from each record set into a DataFrame
for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# Display columns of the main record set
main_record_set_id = record_sets_ids[0]  # Choose the primary, or change if needed
print(f"Fields (columns) in the main record set [{main_record_set_id}]:\n{list(dataframes[main_record_set_id].columns)}")
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes example operations for data preparation.

In [ ]:
# Choose a numeric field to analyze (find a suitable field id using the overview above)
# Example: Use 'cr:CoveragePercent' if present, or another numeric field @id
df = dataframes[main_record_set_id]
numeric_fields = [col for col in df.columns if df[col].dtype in ['float64', 'int64']]
print(f"Numeric fields available: {numeric_fields}")

# Select one numeric field for demonstration (replace with an existing one as discovered)
if numeric_fields:
    numeric_field_id = numeric_fields[0]
else:
    numeric_field_id = df.columns[0]  # fallback
print(f"Using numeric field for EDA: {numeric_field_id}")

threshold = df[numeric_field_id].mean()  # Example threshold, can be customized
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalize the selected numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try grouping by a categorical field if available
categorical_fields = [col for col in df.columns if df[col].dtype == 'object' and col != numeric_field_id]
if categorical_fields:
    group_field = categorical_fields[0]
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"Grouped data by {group_field} (showing first 5 rows):")
    print(grouped_df.head())
else:
    print("No suitable group field identified.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Below is an example using matplotlib to plot the distribution of the selected numeric field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the selected numeric field
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field_id], kde=True)
plt.title(f'Distribution of {numeric_field_id}')
plt.xlabel(numeric_field_id)
plt.ylabel('Frequency')
plt.show()

# Boxplot by a group (if available)
if categorical_fields:
    plt.figure(figsize=(10,4))
    sns.boxplot(x=group_field, y=numeric_field_id, data=df)
    plt.title(f'{numeric_field_id} by {group_field}')
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we loaded and explored the FAIR² dataset using the `mlcroissant` library. We listed the available record sets and fields, loaded the data into DataFrames, and performed simple EDA and visualization using field `@id`s. More advanced analysis and domain-specific insights can be obtained by referencing the dataset documentation and schema fields.